# Canário 4128 | MRBR

| Item | Valor |
|---|---|
| Objeto | ``dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios`` (view) |
| Ambiente | DEV, leitura |
| Schema | `DESCRIBE_MRBR_2.csv`, 70 colunas; clustering: não informado |
| Filtro Datalake | `cod_centro = '4128'` |
| Filtro SAP | Centro 4128; confirmar outros filtros e mesmo período |
| Volume esperado da exploratória | 543 linhas |
| Chave observada no universo | num_doc_faturamento + num_exercicio + num_item_fatura; 84915 linhas e 0 grupos repetidos |
| Campos 100% vazios/nulos no universo sem lacuna explícita | 9 (investigar no SAP, não afirmar defeito) |

**Alertas:** dados pessoais mascarados; view sem equivalência confirmada com filtros da tela SAP; valores DOUBLE exigem tolerância; snapshot pode mudar. **Como usar:** importar e executar Run All. SQL independente e somente leitura. Nunca usar SELECT * para dados. Fonte dos resultados: HTML executado de `MRBR`.

### Índice
1 Metadados; 2 Volume; 3 Centro; 4 Chave; 5 Duplicidade; 6 Preenchimento; 7 Cardinalidade; 8 Domínio; 9 Indicadores; 10 Perfil numérico; 11 Totais; 12 Datas; 13 Códigos; 14 Checksum; 15 Amostra; 16 Distribuição; 17 Freshness; 18 Regras; 19 Amostra ampliada; 20 Resumo; 21 Extração completa.


## 1. Metadados

Conferir o comentário do objeto e os filtros SAP. Para views, DETAIL/HISTORY não são executados.


In [0]:
DESCRIBE EXTENDED `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios`;

## 2. Volume

Reconta o volume 4128 no momento da execução.


In [0]:
SELECT COUNT(*) AS linhas_atuais, 543 AS linhas_esperadas_exploratoria FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128';

## 3. Formato do centro

Mostra eventuais variações com zeros à esquerda, sem misturar no recorte principal.


In [0]:
SELECT `cod_centro` AS centro_literal, LENGTH(TRIM(`cod_centro`)) AS tamanho, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE TRIM(`cod_centro`) RLIKE '^0*4128$' GROUP BY `cod_centro`, LENGTH(TRIM(`cod_centro`)) ORDER BY linhas DESC;

## 4. Granularidade

Confirma a chave medida na exploratória sem concatenação de valores nulos.


In [0]:
WITH d AS (SELECT `num_doc_faturamento`, `num_exercicio`, `num_item_fatura` FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' GROUP BY `num_doc_faturamento`, `num_exercicio`, `num_item_fatura`) SELECT (SELECT COUNT(*) FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128') AS linhas, (SELECT COUNT(*) FROM d) AS chaves_distintas;

## 5. Duplicatas e versões

Distingue duplicatas idênticas de versões por hash da linha inteira.


In [0]:
WITH h AS (SELECT t.`num_doc_faturamento`, t.`num_exercicio`, t.`num_item_fatura`, MD5(TO_JSON(STRUCT(t.`num_exercicio`, t.`num_exerc_periodo_atual`, t.`cod_empresa`, t.`nm_fornecedor`, t.`num_doc_faturamento`, t.`cod_emissor_fatura`, t.`dt_base_prazo_pgto`, t.`dt_lancamento`, t.`qt_dias_desconto_1`, t.`num_doc_referencia`, t.`num_doc_compras`, t.`cod_centro`, t.`cod_material`, t.`num_item_pedido`, t.`cod_grupo_compradores`, t.`qt_pedido`, t.`sg_um_pedido`, t.`vl_montante`, t.`sg_moeda`, t.`vl_diferenca`, t.`sg_moeda_interna`, t.`tp_documento`, t.`cod_bloq_pagamento_log`, t.`nm_usuario`, t.`num_item_fatura`, t.`num_seq_classcont`, t.`num_item_doc_referencia`, t.`cod_bloqueio_pgto`, t.`sg_upp`, t.`tp_condicao`, t.`pct_desconto_2`, t.`pct_desconto_1`, t.`vl_taxa_cambio`, t.`st_fatura`, t.`qt_em_upp`, t.`qt_diferenca`, t.`qt_diferenca_1`, t.`cod_determ_preco`, t.`cod_pais_destino`, t.`cod_pais_procedencia`, t.`ind_nota_existente`, t.`num_pacote`, t.`ind_bloq_qtd`, t.`ind_bloq_qualidade_icone`, t.`ind_bloq_qtd_icone`, t.`ind_bloq_qualidade`, t.`ind_bloq_preco_icone`, t.`ind_bloq_qupp_icone`, t.`ind_bloq_qupp`, t.`ind_bloq_preco`, t.`ind_bloq_montante_icone`, t.`ind_bloq_montante_1`, t.`ind_bloq_montante_2`, t.`ind_bloq_manual_icone`, t.`ind_bloq_manual_1`, t.`ind_bloq_manual_2`, t.`ind_bloq_data_icone`, t.`ind_bloq_data`, t.`num_linha_servico`, t.`num_item_contrato_central`, t.`num_item_contrato_superior`, t.`qt_diferenca_dias`, t.`qt_dias_liquido`, t.`qt_dias_desconto_2`, t.`ind_debito_posterior`, t.`dt_conversao`, t.`num_contrato_central`, t.`num_contrato_basico`, t.`ind_custo_acessorio`, t.`cod_chave_controle_sasq`), MAP('ignoreNullFields','false'))) AS hash_linha FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` t WHERE `cod_centro` = '4128'), d AS (SELECT `num_doc_faturamento`, `num_exercicio`, `num_item_fatura`, COUNT(*) AS linhas, COUNT(DISTINCT hash_linha) AS versoes FROM h GROUP BY `num_doc_faturamento`, `num_exercicio`, `num_item_fatura` HAVING COUNT(*) > 1) SELECT COUNT(*) AS chaves_repetidas, COALESCE(SUM(linhas-versoes),0) AS repeticoes_identicas, COUNT_IF(versoes>1) AS chaves_com_versoes FROM d;

## 6. Preenchimento de todas as colunas

Uma linha por coluna, sem expor valores pessoais. Campos vazios no universo precisam de validação no centro.


In [0]:
SELECT *, total-nulos-vazios AS preenchidos FROM (SELECT 'num_exercicio' AS coluna, COUNT(*) AS total, COUNT_IF(`num_exercicio` IS NULL) AS nulos, COUNT_IF(`num_exercicio` IS NOT NULL AND LOWER(TRIM(`num_exercicio`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`num_exercicio`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'num_exerc_periodo_atual' AS coluna, COUNT(*) AS total, COUNT_IF(`num_exerc_periodo_atual` IS NULL) AS nulos, COUNT_IF(`num_exerc_periodo_atual` IS NOT NULL AND LOWER(TRIM(`num_exerc_periodo_atual`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`num_exerc_periodo_atual`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'cod_empresa' AS coluna, COUNT(*) AS total, COUNT_IF(`cod_empresa` IS NULL) AS nulos, COUNT_IF(`cod_empresa` IS NOT NULL AND LOWER(TRIM(`cod_empresa`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`cod_empresa`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'nm_fornecedor' AS coluna, COUNT(*) AS total, COUNT_IF(`nm_fornecedor` IS NULL) AS nulos, COUNT_IF(`nm_fornecedor` IS NOT NULL AND LOWER(TRIM(`nm_fornecedor`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`nm_fornecedor`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'num_doc_faturamento' AS coluna, COUNT(*) AS total, COUNT_IF(`num_doc_faturamento` IS NULL) AS nulos, COUNT_IF(`num_doc_faturamento` IS NOT NULL AND LOWER(TRIM(`num_doc_faturamento`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`num_doc_faturamento`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'cod_emissor_fatura' AS coluna, COUNT(*) AS total, COUNT_IF(`cod_emissor_fatura` IS NULL) AS nulos, COUNT_IF(`cod_emissor_fatura` IS NOT NULL AND LOWER(TRIM(`cod_emissor_fatura`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`cod_emissor_fatura`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'dt_base_prazo_pgto' AS coluna, COUNT(*) AS total, COUNT_IF(`dt_base_prazo_pgto` IS NULL) AS nulos, COUNT_IF(`dt_base_prazo_pgto` IS NOT NULL AND LOWER(TRIM(`dt_base_prazo_pgto`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`dt_base_prazo_pgto`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'dt_lancamento' AS coluna, COUNT(*) AS total, COUNT_IF(`dt_lancamento` IS NULL) AS nulos, COUNT_IF(`dt_lancamento` IS NOT NULL AND LOWER(TRIM(`dt_lancamento`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`dt_lancamento`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'qt_dias_desconto_1' AS coluna, COUNT(*) AS total, COUNT_IF(`qt_dias_desconto_1` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, COUNT_IF(`qt_dias_desconto_1`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'num_doc_referencia' AS coluna, COUNT(*) AS total, COUNT_IF(`num_doc_referencia` IS NULL) AS nulos, COUNT_IF(`num_doc_referencia` IS NOT NULL AND LOWER(TRIM(`num_doc_referencia`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`num_doc_referencia`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'num_doc_compras' AS coluna, COUNT(*) AS total, COUNT_IF(`num_doc_compras` IS NULL) AS nulos, COUNT_IF(`num_doc_compras` IS NOT NULL AND LOWER(TRIM(`num_doc_compras`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`num_doc_compras`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'cod_centro' AS coluna, COUNT(*) AS total, COUNT_IF(`cod_centro` IS NULL) AS nulos, COUNT_IF(`cod_centro` IS NOT NULL AND LOWER(TRIM(`cod_centro`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`cod_centro`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'cod_material' AS coluna, COUNT(*) AS total, COUNT_IF(`cod_material` IS NULL) AS nulos, COUNT_IF(`cod_material` IS NOT NULL AND LOWER(TRIM(`cod_material`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`cod_material`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'num_item_pedido' AS coluna, COUNT(*) AS total, COUNT_IF(`num_item_pedido` IS NULL) AS nulos, COUNT_IF(`num_item_pedido` IS NOT NULL AND LOWER(TRIM(`num_item_pedido`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`num_item_pedido`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'cod_grupo_compradores' AS coluna, COUNT(*) AS total, COUNT_IF(`cod_grupo_compradores` IS NULL) AS nulos, COUNT_IF(`cod_grupo_compradores` IS NOT NULL AND LOWER(TRIM(`cod_grupo_compradores`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`cod_grupo_compradores`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'qt_pedido' AS coluna, COUNT(*) AS total, COUNT_IF(`qt_pedido` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, COUNT_IF(`qt_pedido`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'sg_um_pedido' AS coluna, COUNT(*) AS total, COUNT_IF(`sg_um_pedido` IS NULL) AS nulos, COUNT_IF(`sg_um_pedido` IS NOT NULL AND LOWER(TRIM(`sg_um_pedido`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`sg_um_pedido`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'vl_montante' AS coluna, COUNT(*) AS total, COUNT_IF(`vl_montante` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, COUNT_IF(`vl_montante`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'sg_moeda' AS coluna, COUNT(*) AS total, COUNT_IF(`sg_moeda` IS NULL) AS nulos, COUNT_IF(`sg_moeda` IS NOT NULL AND LOWER(TRIM(`sg_moeda`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`sg_moeda`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'vl_diferenca' AS coluna, COUNT(*) AS total, COUNT_IF(`vl_diferenca` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, COUNT_IF(`vl_diferenca`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'sg_moeda_interna' AS coluna, COUNT(*) AS total, COUNT_IF(`sg_moeda_interna` IS NULL) AS nulos, COUNT_IF(`sg_moeda_interna` IS NOT NULL AND LOWER(TRIM(`sg_moeda_interna`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`sg_moeda_interna`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'tp_documento' AS coluna, COUNT(*) AS total, COUNT_IF(`tp_documento` IS NULL) AS nulos, COUNT_IF(`tp_documento` IS NOT NULL AND LOWER(TRIM(`tp_documento`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`tp_documento`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'cod_bloq_pagamento_log' AS coluna, COUNT(*) AS total, COUNT_IF(`cod_bloq_pagamento_log` IS NULL) AS nulos, COUNT_IF(`cod_bloq_pagamento_log` IS NOT NULL AND LOWER(TRIM(`cod_bloq_pagamento_log`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`cod_bloq_pagamento_log`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'nm_usuario' AS coluna, COUNT(*) AS total, COUNT_IF(`nm_usuario` IS NULL) AS nulos, COUNT_IF(`nm_usuario` IS NOT NULL AND LOWER(TRIM(`nm_usuario`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`nm_usuario`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'num_item_fatura' AS coluna, COUNT(*) AS total, COUNT_IF(`num_item_fatura` IS NULL) AS nulos, COUNT_IF(`num_item_fatura` IS NOT NULL AND LOWER(TRIM(`num_item_fatura`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`num_item_fatura`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'num_seq_classcont' AS coluna, COUNT(*) AS total, COUNT_IF(`num_seq_classcont` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, COUNT_IF(`num_seq_classcont`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'num_item_doc_referencia' AS coluna, COUNT(*) AS total, COUNT_IF(`num_item_doc_referencia` IS NULL) AS nulos, COUNT_IF(`num_item_doc_referencia` IS NOT NULL AND LOWER(TRIM(`num_item_doc_referencia`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`num_item_doc_referencia`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'cod_bloqueio_pgto' AS coluna, COUNT(*) AS total, COUNT_IF(`cod_bloqueio_pgto` IS NULL) AS nulos, COUNT_IF(`cod_bloqueio_pgto` IS NOT NULL AND LOWER(TRIM(`cod_bloqueio_pgto`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`cod_bloqueio_pgto`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'sg_upp' AS coluna, COUNT(*) AS total, COUNT_IF(`sg_upp` IS NULL) AS nulos, COUNT_IF(`sg_upp` IS NOT NULL AND LOWER(TRIM(`sg_upp`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`sg_upp`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'tp_condicao' AS coluna, COUNT(*) AS total, COUNT_IF(`tp_condicao` IS NULL) AS nulos, COUNT_IF(`tp_condicao` IS NOT NULL AND LOWER(TRIM(`tp_condicao`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`tp_condicao`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'pct_desconto_2' AS coluna, COUNT(*) AS total, COUNT_IF(`pct_desconto_2` IS NULL) AS nulos, COUNT_IF(`pct_desconto_2` IS NOT NULL AND LOWER(TRIM(`pct_desconto_2`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`pct_desconto_2`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'pct_desconto_1' AS coluna, COUNT(*) AS total, COUNT_IF(`pct_desconto_1` IS NULL) AS nulos, COUNT_IF(`pct_desconto_1` IS NOT NULL AND LOWER(TRIM(`pct_desconto_1`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`pct_desconto_1`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'vl_taxa_cambio' AS coluna, COUNT(*) AS total, COUNT_IF(`vl_taxa_cambio` IS NULL) AS nulos, COUNT_IF(`vl_taxa_cambio` IS NOT NULL AND LOWER(TRIM(`vl_taxa_cambio`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`vl_taxa_cambio`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'st_fatura' AS coluna, COUNT(*) AS total, COUNT_IF(`st_fatura` IS NULL) AS nulos, COUNT_IF(`st_fatura` IS NOT NULL AND LOWER(TRIM(`st_fatura`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`st_fatura`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'qt_em_upp' AS coluna, COUNT(*) AS total, COUNT_IF(`qt_em_upp` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, COUNT_IF(`qt_em_upp`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'qt_diferenca' AS coluna, COUNT(*) AS total, COUNT_IF(`qt_diferenca` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, COUNT_IF(`qt_diferenca`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'qt_diferenca_1' AS coluna, COUNT(*) AS total, COUNT_IF(`qt_diferenca_1` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, COUNT_IF(`qt_diferenca_1`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'cod_determ_preco' AS coluna, COUNT(*) AS total, COUNT_IF(`cod_determ_preco` IS NULL) AS nulos, COUNT_IF(`cod_determ_preco` IS NOT NULL AND LOWER(TRIM(`cod_determ_preco`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`cod_determ_preco`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'cod_pais_destino' AS coluna, COUNT(*) AS total, COUNT_IF(`cod_pais_destino` IS NULL) AS nulos, COUNT_IF(`cod_pais_destino` IS NOT NULL AND LOWER(TRIM(`cod_pais_destino`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`cod_pais_destino`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'cod_pais_procedencia' AS coluna, COUNT(*) AS total, COUNT_IF(`cod_pais_procedencia` IS NULL) AS nulos, COUNT_IF(`cod_pais_procedencia` IS NOT NULL AND LOWER(TRIM(`cod_pais_procedencia`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`cod_pais_procedencia`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'ind_nota_existente' AS coluna, COUNT(*) AS total, COUNT_IF(`ind_nota_existente` IS NULL) AS nulos, COUNT_IF(`ind_nota_existente` IS NOT NULL AND LOWER(TRIM(`ind_nota_existente`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`ind_nota_existente`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'num_pacote' AS coluna, COUNT(*) AS total, COUNT_IF(`num_pacote` IS NULL) AS nulos, COUNT_IF(`num_pacote` IS NOT NULL AND LOWER(TRIM(`num_pacote`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`num_pacote`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'ind_bloq_qtd' AS coluna, COUNT(*) AS total, COUNT_IF(`ind_bloq_qtd` IS NULL) AS nulos, COUNT_IF(`ind_bloq_qtd` IS NOT NULL AND LOWER(TRIM(`ind_bloq_qtd`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`ind_bloq_qtd`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'ind_bloq_qualidade_icone' AS coluna, COUNT(*) AS total, COUNT_IF(`ind_bloq_qualidade_icone` IS NULL) AS nulos, COUNT_IF(`ind_bloq_qualidade_icone` IS NOT NULL AND LOWER(TRIM(`ind_bloq_qualidade_icone`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`ind_bloq_qualidade_icone`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'ind_bloq_qtd_icone' AS coluna, COUNT(*) AS total, COUNT_IF(`ind_bloq_qtd_icone` IS NULL) AS nulos, COUNT_IF(`ind_bloq_qtd_icone` IS NOT NULL AND LOWER(TRIM(`ind_bloq_qtd_icone`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`ind_bloq_qtd_icone`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'ind_bloq_qualidade' AS coluna, COUNT(*) AS total, COUNT_IF(`ind_bloq_qualidade` IS NULL) AS nulos, COUNT_IF(`ind_bloq_qualidade` IS NOT NULL AND LOWER(TRIM(`ind_bloq_qualidade`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`ind_bloq_qualidade`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'ind_bloq_preco_icone' AS coluna, COUNT(*) AS total, COUNT_IF(`ind_bloq_preco_icone` IS NULL) AS nulos, COUNT_IF(`ind_bloq_preco_icone` IS NOT NULL AND LOWER(TRIM(`ind_bloq_preco_icone`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`ind_bloq_preco_icone`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'ind_bloq_qupp_icone' AS coluna, COUNT(*) AS total, COUNT_IF(`ind_bloq_qupp_icone` IS NULL) AS nulos, COUNT_IF(`ind_bloq_qupp_icone` IS NOT NULL AND LOWER(TRIM(`ind_bloq_qupp_icone`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`ind_bloq_qupp_icone`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'ind_bloq_qupp' AS coluna, COUNT(*) AS total, COUNT_IF(`ind_bloq_qupp` IS NULL) AS nulos, COUNT_IF(`ind_bloq_qupp` IS NOT NULL AND LOWER(TRIM(`ind_bloq_qupp`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`ind_bloq_qupp`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'ind_bloq_preco' AS coluna, COUNT(*) AS total, COUNT_IF(`ind_bloq_preco` IS NULL) AS nulos, COUNT_IF(`ind_bloq_preco` IS NOT NULL AND LOWER(TRIM(`ind_bloq_preco`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`ind_bloq_preco`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'ind_bloq_montante_icone' AS coluna, COUNT(*) AS total, COUNT_IF(`ind_bloq_montante_icone` IS NULL) AS nulos, COUNT_IF(`ind_bloq_montante_icone` IS NOT NULL AND LOWER(TRIM(`ind_bloq_montante_icone`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`ind_bloq_montante_icone`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'ind_bloq_montante_1' AS coluna, COUNT(*) AS total, COUNT_IF(`ind_bloq_montante_1` IS NULL) AS nulos, COUNT_IF(`ind_bloq_montante_1` IS NOT NULL AND LOWER(TRIM(`ind_bloq_montante_1`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`ind_bloq_montante_1`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'ind_bloq_montante_2' AS coluna, COUNT(*) AS total, COUNT_IF(`ind_bloq_montante_2` IS NULL) AS nulos, COUNT_IF(`ind_bloq_montante_2` IS NOT NULL AND LOWER(TRIM(`ind_bloq_montante_2`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`ind_bloq_montante_2`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'ind_bloq_manual_icone' AS coluna, COUNT(*) AS total, COUNT_IF(`ind_bloq_manual_icone` IS NULL) AS nulos, COUNT_IF(`ind_bloq_manual_icone` IS NOT NULL AND LOWER(TRIM(`ind_bloq_manual_icone`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`ind_bloq_manual_icone`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'ind_bloq_manual_1' AS coluna, COUNT(*) AS total, COUNT_IF(`ind_bloq_manual_1` IS NULL) AS nulos, COUNT_IF(`ind_bloq_manual_1` IS NOT NULL AND LOWER(TRIM(`ind_bloq_manual_1`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`ind_bloq_manual_1`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'ind_bloq_manual_2' AS coluna, COUNT(*) AS total, COUNT_IF(`ind_bloq_manual_2` IS NULL) AS nulos, COUNT_IF(`ind_bloq_manual_2` IS NOT NULL AND LOWER(TRIM(`ind_bloq_manual_2`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`ind_bloq_manual_2`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'ind_bloq_data_icone' AS coluna, COUNT(*) AS total, COUNT_IF(`ind_bloq_data_icone` IS NULL) AS nulos, COUNT_IF(`ind_bloq_data_icone` IS NOT NULL AND LOWER(TRIM(`ind_bloq_data_icone`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`ind_bloq_data_icone`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'ind_bloq_data' AS coluna, COUNT(*) AS total, COUNT_IF(`ind_bloq_data` IS NULL) AS nulos, COUNT_IF(`ind_bloq_data` IS NOT NULL AND LOWER(TRIM(`ind_bloq_data`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`ind_bloq_data`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'num_linha_servico' AS coluna, COUNT(*) AS total, COUNT_IF(`num_linha_servico` IS NULL) AS nulos, COUNT_IF(`num_linha_servico` IS NOT NULL AND LOWER(TRIM(`num_linha_servico`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`num_linha_servico`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'num_item_contrato_central' AS coluna, COUNT(*) AS total, COUNT_IF(`num_item_contrato_central` IS NULL) AS nulos, COUNT_IF(`num_item_contrato_central` IS NOT NULL AND LOWER(TRIM(`num_item_contrato_central`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`num_item_contrato_central`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'num_item_contrato_superior' AS coluna, COUNT(*) AS total, COUNT_IF(`num_item_contrato_superior` IS NULL) AS nulos, COUNT_IF(`num_item_contrato_superior` IS NOT NULL AND LOWER(TRIM(`num_item_contrato_superior`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`num_item_contrato_superior`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'qt_diferenca_dias' AS coluna, COUNT(*) AS total, COUNT_IF(`qt_diferenca_dias` IS NULL) AS nulos, COUNT_IF(`qt_diferenca_dias` IS NOT NULL AND LOWER(TRIM(`qt_diferenca_dias`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`qt_diferenca_dias`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'qt_dias_liquido' AS coluna, COUNT(*) AS total, COUNT_IF(`qt_dias_liquido` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, COUNT_IF(`qt_dias_liquido`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'qt_dias_desconto_2' AS coluna, COUNT(*) AS total, COUNT_IF(`qt_dias_desconto_2` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, COUNT_IF(`qt_dias_desconto_2`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'ind_debito_posterior' AS coluna, COUNT(*) AS total, COUNT_IF(`ind_debito_posterior` IS NULL) AS nulos, COUNT_IF(`ind_debito_posterior` IS NOT NULL AND LOWER(TRIM(`ind_debito_posterior`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`ind_debito_posterior`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'dt_conversao' AS coluna, COUNT(*) AS total, COUNT_IF(`dt_conversao` IS NULL) AS nulos, COUNT_IF(`dt_conversao` IS NOT NULL AND LOWER(TRIM(`dt_conversao`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`dt_conversao`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'num_contrato_central' AS coluna, COUNT(*) AS total, COUNT_IF(`num_contrato_central` IS NULL) AS nulos, COUNT_IF(`num_contrato_central` IS NOT NULL AND LOWER(TRIM(`num_contrato_central`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`num_contrato_central`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'num_contrato_basico' AS coluna, COUNT(*) AS total, COUNT_IF(`num_contrato_basico` IS NULL) AS nulos, COUNT_IF(`num_contrato_basico` IS NOT NULL AND LOWER(TRIM(`num_contrato_basico`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`num_contrato_basico`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'ind_custo_acessorio' AS coluna, COUNT(*) AS total, COUNT_IF(`ind_custo_acessorio` IS NULL) AS nulos, COUNT_IF(`ind_custo_acessorio` IS NOT NULL AND LOWER(TRIM(`ind_custo_acessorio`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`ind_custo_acessorio`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'cod_chave_controle_sasq' AS coluna, COUNT(*) AS total, COUNT_IF(`cod_chave_controle_sasq` IS NULL) AS nulos, COUNT_IF(`cod_chave_controle_sasq` IS NOT NULL AND LOWER(TRIM(`cod_chave_controle_sasq`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`cod_chave_controle_sasq`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128') a ORDER BY coluna;

## 6.1. Campos com lacuna documentada

Verifica campos explicitamente documentados como sem fonte ou vazios.


In [0]:
SELECT * FROM (SELECT 'cod_chave_controle_sasq' AS coluna, COUNT(*) AS total, COUNT_IF(`cod_chave_controle_sasq` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'cod_determ_preco' AS coluna, COUNT(*) AS total, COUNT_IF(`cod_determ_preco` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'ind_bloq_data_icone' AS coluna, COUNT(*) AS total, COUNT_IF(`ind_bloq_data_icone` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'ind_bloq_manual_icone' AS coluna, COUNT(*) AS total, COUNT_IF(`ind_bloq_manual_icone` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'ind_bloq_montante_icone' AS coluna, COUNT(*) AS total, COUNT_IF(`ind_bloq_montante_icone` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'ind_bloq_preco_icone' AS coluna, COUNT(*) AS total, COUNT_IF(`ind_bloq_preco_icone` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'ind_bloq_qtd_icone' AS coluna, COUNT(*) AS total, COUNT_IF(`ind_bloq_qtd_icone` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'ind_bloq_qualidade_icone' AS coluna, COUNT(*) AS total, COUNT_IF(`ind_bloq_qualidade_icone` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'ind_bloq_qupp_icone' AS coluna, COUNT(*) AS total, COUNT_IF(`ind_bloq_qupp_icone` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'ind_nota_existente' AS coluna, COUNT(*) AS total, COUNT_IF(`ind_nota_existente` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'num_contrato_central' AS coluna, COUNT(*) AS total, COUNT_IF(`num_contrato_central` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'num_item_contrato_central' AS coluna, COUNT(*) AS total, COUNT_IF(`num_item_contrato_central` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'num_linha_servico' AS coluna, COUNT(*) AS total, COUNT_IF(`num_linha_servico` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'num_pacote' AS coluna, COUNT(*) AS total, COUNT_IF(`num_pacote` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'qt_diferenca' AS coluna, COUNT(*) AS total, COUNT_IF(`qt_diferenca` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'qt_diferenca_1' AS coluna, COUNT(*) AS total, COUNT_IF(`qt_diferenca_1` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'qt_diferenca_dias' AS coluna, COUNT(*) AS total, COUNT_IF(`qt_diferenca_dias` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'st_fatura' AS coluna, COUNT(*) AS total, COUNT_IF(`st_fatura` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128') a ORDER BY coluna;

## 7. Cardinalidade

Compara a cardinalidade dos identificadores e categorias não pessoais.


In [0]:
SELECT * FROM (SELECT 'num_doc_faturamento' AS coluna, COUNT(DISTINCT `num_doc_faturamento`) AS distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'num_exercicio' AS coluna, COUNT(DISTINCT `num_exercicio`) AS distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'num_item_fatura' AS coluna, COUNT(DISTINCT `num_item_fatura`) AS distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'cod_centro' AS coluna, COUNT(DISTINCT `cod_centro`) AS distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'sg_um_pedido' AS coluna, COUNT(DISTINCT `sg_um_pedido`) AS distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'sg_moeda' AS coluna, COUNT(DISTINCT `sg_moeda`) AS distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'sg_moeda_interna' AS coluna, COUNT(DISTINCT `sg_moeda_interna`) AS distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'tp_documento' AS coluna, COUNT(DISTINCT `tp_documento`) AS distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128') a ORDER BY coluna;

## 8. Domínio

Oito valores mais comuns por categoria não pessoal.


In [0]:
WITH d AS (SELECT 'sg_um_pedido' AS coluna, CAST(`sg_um_pedido` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' GROUP BY `sg_um_pedido` UNION ALL SELECT 'sg_moeda' AS coluna, CAST(`sg_moeda` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' GROUP BY `sg_moeda` UNION ALL SELECT 'sg_moeda_interna' AS coluna, CAST(`sg_moeda_interna` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' GROUP BY `sg_moeda_interna` UNION ALL SELECT 'tp_documento' AS coluna, CAST(`tp_documento` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' GROUP BY `tp_documento` UNION ALL SELECT 'cod_bloq_pagamento_log' AS coluna, CAST(`cod_bloq_pagamento_log` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' GROUP BY `cod_bloq_pagamento_log` UNION ALL SELECT 'cod_bloqueio_pgto' AS coluna, CAST(`cod_bloqueio_pgto` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' GROUP BY `cod_bloqueio_pgto`) SELECT coluna, valor, linhas FROM (SELECT d.*, ROW_NUMBER() OVER(PARTITION BY coluna ORDER BY linhas DESC, valor) AS rn FROM d) x WHERE rn<=8 ORDER BY coluna,rn;

## 9. Indicadores

Sem indicadores selecionados.


In [0]:
SELECT 'Sem indicadores selecionados' AS observacao;

## 10. Perfil numérico

Quantidades e valores sem tratar códigos como medidas.


In [0]:
SELECT * FROM (SELECT 'qt_dias_desconto_1' AS coluna, CAST(MIN(`qt_dias_desconto_1`) AS DOUBLE) AS minimo, CAST(MAX(`qt_dias_desconto_1`) AS DOUBLE) AS maximo, CAST(AVG(`qt_dias_desconto_1`) AS DOUBLE) AS media, CAST(PERCENTILE_APPROX(`qt_dias_desconto_1`,0.5) AS DOUBLE) AS mediana, COUNT_IF(`qt_dias_desconto_1`<0) AS negativos, COUNT_IF(`qt_dias_desconto_1`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'qt_pedido' AS coluna, CAST(MIN(`qt_pedido`) AS DOUBLE) AS minimo, CAST(MAX(`qt_pedido`) AS DOUBLE) AS maximo, CAST(AVG(`qt_pedido`) AS DOUBLE) AS media, CAST(PERCENTILE_APPROX(`qt_pedido`,0.5) AS DOUBLE) AS mediana, COUNT_IF(`qt_pedido`<0) AS negativos, COUNT_IF(`qt_pedido`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'vl_montante' AS coluna, CAST(MIN(`vl_montante`) AS DOUBLE) AS minimo, CAST(MAX(`vl_montante`) AS DOUBLE) AS maximo, CAST(AVG(`vl_montante`) AS DOUBLE) AS media, CAST(PERCENTILE_APPROX(`vl_montante`,0.5) AS DOUBLE) AS mediana, COUNT_IF(`vl_montante`<0) AS negativos, COUNT_IF(`vl_montante`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'vl_diferenca' AS coluna, CAST(MIN(`vl_diferenca`) AS DOUBLE) AS minimo, CAST(MAX(`vl_diferenca`) AS DOUBLE) AS maximo, CAST(AVG(`vl_diferenca`) AS DOUBLE) AS media, CAST(PERCENTILE_APPROX(`vl_diferenca`,0.5) AS DOUBLE) AS mediana, COUNT_IF(`vl_diferenca`<0) AS negativos, COUNT_IF(`vl_diferenca`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'qt_em_upp' AS coluna, CAST(MIN(`qt_em_upp`) AS DOUBLE) AS minimo, CAST(MAX(`qt_em_upp`) AS DOUBLE) AS maximo, CAST(AVG(`qt_em_upp`) AS DOUBLE) AS media, CAST(PERCENTILE_APPROX(`qt_em_upp`,0.5) AS DOUBLE) AS mediana, COUNT_IF(`qt_em_upp`<0) AS negativos, COUNT_IF(`qt_em_upp`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'qt_diferenca' AS coluna, CAST(MIN(`qt_diferenca`) AS DOUBLE) AS minimo, CAST(MAX(`qt_diferenca`) AS DOUBLE) AS maximo, CAST(AVG(`qt_diferenca`) AS DOUBLE) AS media, CAST(PERCENTILE_APPROX(`qt_diferenca`,0.5) AS DOUBLE) AS mediana, COUNT_IF(`qt_diferenca`<0) AS negativos, COUNT_IF(`qt_diferenca`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'qt_diferenca_1' AS coluna, CAST(MIN(`qt_diferenca_1`) AS DOUBLE) AS minimo, CAST(MAX(`qt_diferenca_1`) AS DOUBLE) AS maximo, CAST(AVG(`qt_diferenca_1`) AS DOUBLE) AS media, CAST(PERCENTILE_APPROX(`qt_diferenca_1`,0.5) AS DOUBLE) AS mediana, COUNT_IF(`qt_diferenca_1`<0) AS negativos, COUNT_IF(`qt_diferenca_1`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'qt_dias_liquido' AS coluna, CAST(MIN(`qt_dias_liquido`) AS DOUBLE) AS minimo, CAST(MAX(`qt_dias_liquido`) AS DOUBLE) AS maximo, CAST(AVG(`qt_dias_liquido`) AS DOUBLE) AS media, CAST(PERCENTILE_APPROX(`qt_dias_liquido`,0.5) AS DOUBLE) AS mediana, COUNT_IF(`qt_dias_liquido`<0) AS negativos, COUNT_IF(`qt_dias_liquido`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'qt_dias_desconto_2' AS coluna, CAST(MIN(`qt_dias_desconto_2`) AS DOUBLE) AS minimo, CAST(MAX(`qt_dias_desconto_2`) AS DOUBLE) AS maximo, CAST(AVG(`qt_dias_desconto_2`) AS DOUBLE) AS media, CAST(PERCENTILE_APPROX(`qt_dias_desconto_2`,0.5) AS DOUBLE) AS mediana, COUNT_IF(`qt_dias_desconto_2`<0) AS negativos, COUNT_IF(`qt_dias_desconto_2`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128') a ORDER BY coluna;

## 11. Totais para conciliação

Totais por campo; verificar moeda, granularidade e DOUBLE.


In [0]:
SELECT * FROM (SELECT 'qt_dias_desconto_1' AS coluna, CAST(SUM(`qt_dias_desconto_1`) AS DOUBLE) AS total FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'qt_pedido' AS coluna, CAST(SUM(`qt_pedido`) AS DOUBLE) AS total FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'vl_montante' AS coluna, CAST(SUM(`vl_montante`) AS DOUBLE) AS total FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'vl_diferenca' AS coluna, CAST(SUM(`vl_diferenca`) AS DOUBLE) AS total FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'qt_em_upp' AS coluna, CAST(SUM(`qt_em_upp`) AS DOUBLE) AS total FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'qt_diferenca' AS coluna, CAST(SUM(`qt_diferenca`) AS DOUBLE) AS total FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'qt_diferenca_1' AS coluna, CAST(SUM(`qt_diferenca_1`) AS DOUBLE) AS total FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'qt_dias_liquido' AS coluna, CAST(SUM(`qt_dias_liquido`) AS DOUBLE) AS total FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'qt_dias_desconto_2' AS coluna, CAST(SUM(`qt_dias_desconto_2`) AS DOUBLE) AS total FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128') a ORDER BY coluna;

## 12. Datas

Min/max de strings são literais, não ordem cronológica.


In [0]:
SELECT * FROM (SELECT 'dt_base_prazo_pgto' AS coluna, CAST(MIN(`dt_base_prazo_pgto`) AS STRING) AS menor_literal, CAST(MAX(`dt_base_prazo_pgto`) AS STRING) AS maior_literal, COUNT_IF(TRIM(`dt_base_prazo_pgto`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}$') AS formato_dd_mm_aaaa FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'dt_lancamento' AS coluna, CAST(MIN(`dt_lancamento`) AS STRING) AS menor_literal, CAST(MAX(`dt_lancamento`) AS STRING) AS maior_literal, COUNT_IF(TRIM(`dt_lancamento`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}$') AS formato_dd_mm_aaaa FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'dt_conversao' AS coluna, CAST(MIN(`dt_conversao`) AS STRING) AS menor_literal, CAST(MAX(`dt_conversao`) AS STRING) AS maior_literal, COUNT_IF(TRIM(`dt_conversao`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}$') AS formato_dd_mm_aaaa FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128') a ORDER BY coluna;

## 13. Zeros à esquerda

Mede códigos numéricos com prefixo zero sem alterar a fonte.


In [0]:
SELECT * FROM (SELECT 'num_exercicio' AS coluna, COUNT_IF(TRIM(`num_exercicio`) RLIKE '^0+[0-9]+$') AS com_zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'num_exerc_periodo_atual' AS coluna, COUNT_IF(TRIM(`num_exerc_periodo_atual`) RLIKE '^0+[0-9]+$') AS com_zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'cod_empresa' AS coluna, COUNT_IF(TRIM(`cod_empresa`) RLIKE '^0+[0-9]+$') AS com_zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'num_doc_faturamento' AS coluna, COUNT_IF(TRIM(`num_doc_faturamento`) RLIKE '^0+[0-9]+$') AS com_zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'cod_emissor_fatura' AS coluna, COUNT_IF(TRIM(`cod_emissor_fatura`) RLIKE '^0+[0-9]+$') AS com_zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' UNION ALL SELECT 'num_doc_referencia' AS coluna, COUNT_IF(TRIM(`num_doc_referencia`) RLIKE '^0+[0-9]+$') AS com_zeros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128') a ORDER BY coluna;

## 14. Checksum

Duplicatas idênticas sem divulgar hash ou dados.


In [0]:
WITH h AS (SELECT MD5(TO_JSON(STRUCT(t.`num_exercicio`, t.`num_exerc_periodo_atual`, t.`cod_empresa`, t.`nm_fornecedor`, t.`num_doc_faturamento`, t.`cod_emissor_fatura`, t.`dt_base_prazo_pgto`, t.`dt_lancamento`, t.`qt_dias_desconto_1`, t.`num_doc_referencia`, t.`num_doc_compras`, t.`cod_centro`, t.`cod_material`, t.`num_item_pedido`, t.`cod_grupo_compradores`, t.`qt_pedido`, t.`sg_um_pedido`, t.`vl_montante`, t.`sg_moeda`, t.`vl_diferenca`, t.`sg_moeda_interna`, t.`tp_documento`, t.`cod_bloq_pagamento_log`, t.`nm_usuario`, t.`num_item_fatura`, t.`num_seq_classcont`, t.`num_item_doc_referencia`, t.`cod_bloqueio_pgto`, t.`sg_upp`, t.`tp_condicao`, t.`pct_desconto_2`, t.`pct_desconto_1`, t.`vl_taxa_cambio`, t.`st_fatura`, t.`qt_em_upp`, t.`qt_diferenca`, t.`qt_diferenca_1`, t.`cod_determ_preco`, t.`cod_pais_destino`, t.`cod_pais_procedencia`, t.`ind_nota_existente`, t.`num_pacote`, t.`ind_bloq_qtd`, t.`ind_bloq_qualidade_icone`, t.`ind_bloq_qtd_icone`, t.`ind_bloq_qualidade`, t.`ind_bloq_preco_icone`, t.`ind_bloq_qupp_icone`, t.`ind_bloq_qupp`, t.`ind_bloq_preco`, t.`ind_bloq_montante_icone`, t.`ind_bloq_montante_1`, t.`ind_bloq_montante_2`, t.`ind_bloq_manual_icone`, t.`ind_bloq_manual_1`, t.`ind_bloq_manual_2`, t.`ind_bloq_data_icone`, t.`ind_bloq_data`, t.`num_linha_servico`, t.`num_item_contrato_central`, t.`num_item_contrato_superior`, t.`qt_diferenca_dias`, t.`qt_dias_liquido`, t.`qt_dias_desconto_2`, t.`ind_debito_posterior`, t.`dt_conversao`, t.`num_contrato_central`, t.`num_contrato_basico`, t.`ind_custo_acessorio`, t.`cod_chave_controle_sasq`), MAP('ignoreNullFields','false'))) AS hash_linha FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` t WHERE `cod_centro` = '4128'), d AS (SELECT hash_linha,COUNT(*) AS linhas FROM h GROUP BY hash_linha) SELECT COALESCE(SUM(linhas),0) AS linhas, COUNT(*) AS distintos, COALESCE(SUM(linhas-1),0) AS duplicatas_identicas FROM d;

## 15. Amostra

Sem dados pessoais literais.


In [0]:
SELECT `num_exercicio`, `num_exerc_periodo_atual`, `cod_empresa`, `num_doc_faturamento`, `cod_emissor_fatura`, `dt_base_prazo_pgto`, `dt_lancamento`, `qt_dias_desconto_1`, `num_doc_referencia`, `num_doc_compras`, `cod_centro`, `cod_material`, `num_item_pedido`, `cod_grupo_compradores`, `qt_pedido`, `sg_um_pedido`, `vl_montante`, `sg_moeda`, `vl_diferenca`, `sg_moeda_interna`, SUBSTRING(MD5(CAST(`nm_fornecedor` AS STRING)),1,8) AS `nm_fornecedor_hash`, SUBSTRING(MD5(CAST(`nm_usuario` AS STRING)),1,8) AS `nm_usuario_hash` FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' LIMIT 20;

## 16. Distribuição

Dimensões de negócio sem pessoas.


In [0]:
SELECT `cod_centro`, `sg_um_pedido`, `sg_moeda`,COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' GROUP BY `cod_centro`, `sg_um_pedido`, `sg_moeda` ORDER BY linhas DESC LIMIT 100;

## 17. Freshness

Sem coluna de ingestão.


In [0]:
SELECT CURRENT_TIMESTAMP() AS instante,COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128';

## 18. Regras específicas

Testes derivados do DESCRIBE, não conclusões.


In [0]:
SELECT COUNT(*) AS linhas,COUNT_IF(`vl_diferenca` IS NULL) AS diferenca_nula,SUM(`vl_montante`) AS montante FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128';

## 19. Amostra ampliada

Até 500 linhas sem dados pessoais literais.


In [0]:
SELECT `num_exercicio`, `num_exerc_periodo_atual`, `cod_empresa`, `num_doc_faturamento`, `cod_emissor_fatura`, `dt_base_prazo_pgto`, `dt_lancamento`, `qt_dias_desconto_1`, `num_doc_referencia`, `num_doc_compras`, `cod_centro`, `cod_material`, `num_item_pedido`, `cod_grupo_compradores`, `qt_pedido`, `sg_um_pedido`, `vl_montante`, `sg_moeda`, `vl_diferenca`, `sg_moeda_interna`, SUBSTRING(MD5(CAST(`nm_fornecedor` AS STRING)),1,8) AS `nm_fornecedor_hash`, SUBSTRING(MD5(CAST(`nm_usuario` AS STRING)),1,8) AS `nm_usuario_hash` FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' LIMIT 500;

## 20. Resumo

Volume atual e chave observada no universo.


In [0]:
WITH d AS (SELECT `num_doc_faturamento`, `num_exercicio`, `num_item_fatura`,COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' GROUP BY `num_doc_faturamento`, `num_exercicio`, `num_item_fatura`) SELECT 'MRBR' AS transacao,'4128' AS centro,543 AS volume_esperado,(SELECT COUNT(*) FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128') AS volume_atual,(SELECT COUNT(*) FROM d) AS chaves_distintas,(SELECT COUNT(*) FROM d WHERE linhas>1) AS chaves_repetidas,'num_doc_faturamento + num_exercicio + num_item_fatura' AS chave_observada;

## 21. EXTRAÇÃO COMPLETA da base 4128

Sem LIMIT, sem dados pessoais literais. Comparar com SAP apenas após confirmar filtros da tela.


In [0]:
SELECT CASE WHEN TRIM(CAST(`num_doc_faturamento` AS STRING)) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(CAST(`num_doc_faturamento` AS STRING)),'^0+(?=[0-9])','') ELSE TRIM(CAST(`num_doc_faturamento` AS STRING)) END AS `num_doc_faturamento_normalizado`, CASE WHEN TRIM(CAST(`num_exercicio` AS STRING)) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(CAST(`num_exercicio` AS STRING)),'^0+(?=[0-9])','') ELSE TRIM(CAST(`num_exercicio` AS STRING)) END AS `num_exercicio_normalizado`, CASE WHEN TRIM(CAST(`num_item_fatura` AS STRING)) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(CAST(`num_item_fatura` AS STRING)),'^0+(?=[0-9])','') ELSE TRIM(CAST(`num_item_fatura` AS STRING)) END AS `num_item_fatura_normalizado`, `num_exercicio`, `num_exerc_periodo_atual`, `cod_empresa`, `num_doc_faturamento`, `cod_emissor_fatura`, `dt_base_prazo_pgto`, `dt_lancamento`, `qt_dias_desconto_1`, `num_doc_referencia`, `num_doc_compras`, `cod_centro`, `cod_material`, `num_item_pedido`, `cod_grupo_compradores`, `qt_pedido`, `sg_um_pedido`, `vl_montante`, `sg_moeda`, `vl_diferenca`, `sg_moeda_interna`, `tp_documento`, `cod_bloq_pagamento_log`, `num_item_fatura`, `num_seq_classcont`, `num_item_doc_referencia`, `cod_bloqueio_pgto`, `sg_upp`, `tp_condicao`, `pct_desconto_2`, `pct_desconto_1`, `vl_taxa_cambio`, `st_fatura`, `qt_em_upp`, `qt_diferenca`, `qt_diferenca_1`, `cod_determ_preco`, `cod_pais_destino`, `cod_pais_procedencia`, `ind_nota_existente`, `num_pacote`, `ind_bloq_qtd`, `ind_bloq_qualidade_icone`, `ind_bloq_qtd_icone`, `ind_bloq_qualidade`, `ind_bloq_preco_icone`, `ind_bloq_qupp_icone`, `ind_bloq_qupp`, `ind_bloq_preco`, `ind_bloq_montante_icone`, `ind_bloq_montante_1`, `ind_bloq_montante_2`, `ind_bloq_manual_icone`, `ind_bloq_manual_1`, `ind_bloq_manual_2`, `ind_bloq_data_icone`, `ind_bloq_data`, `num_linha_servico`, `num_item_contrato_central`, `num_item_contrato_superior`, `qt_diferenca_dias`, `qt_dias_liquido`, `qt_dias_desconto_2`, `ind_debito_posterior`, `dt_conversao`, `num_contrato_central`, `num_contrato_basico`, `ind_custo_acessorio`, `cod_chave_controle_sasq`, SUBSTRING(MD5(CAST(`nm_fornecedor` AS STRING)),1,8) AS `nm_fornecedor_hash`, SUBSTRING(MD5(CAST(`nm_usuario` AS STRING)),1,8) AS `nm_usuario_hash` FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE `cod_centro` = '4128' ORDER BY `num_doc_faturamento`, `num_exercicio`, `num_item_fatura`;

### Filtros da tela SAP pendentes

A extração completa acima é da base 4128. Não há filtros adicionais da tela SAP confirmados nos insumos; não foi criada extração supostamente equivalente sem essa definição.


## Checklist SAP

- [ ] Confirmar centro 4128, janela temporal e filtros da tela.
- [ ] Executar no mesmo dia da extração SAP.
- [ ] Conferir chave, duplicatas, zeros à esquerda, datas, moeda e DOUBLE.
- [ ] Validar campos 100% sem valor com o dono do dado.
- [ ] Não exportar dados pessoais literais.
